# Functional Annotation and Pathway Enrichment
## TL;DR — Plain English Introduction

**What you will learn:** How to go from a list of genes to biological meaning — annotating genes with function (GO terms, KEGG pathways) and testing whether specific pathways are enriched in your hit list.

**Why it matters:** The final step in any genomics analysis. Raw DE gene lists are meaningless without biological context. Pathway analysis answers: "What biology is changing?"


## Pathway Enrichment — Concepts for Beginners

| Term | Plain English |
|------|---------------|
| **pathway** | A set of genes/proteins that work together for a specific biological process |
| **GO term** | Gene Ontology term — a standardised label describing a gene's function (e.g. GO:0006915 = apoptosis) |
| **enrichment** | A pathway is enriched if more of your DEGs belong to it than expected by chance |
| **Fisher's exact test** | Statistical test asking: "Is the overlap between my gene list and this pathway surprising?" |
| **hypergeometric test** | Same idea as Fisher's but phrased as drawing balls from an urn — equivalent result |
| **background gene set** | All genes measured in your experiment — the universe from which your DEGs came |
| **GSEA** | Gene Set Enrichment Analysis — ranks ALL genes by fold-change and finds enriched pathways at the top/bottom |
| **KEGG** | Kyoto Encyclopedia of Genes and Genomes — database of metabolic and signalling pathways |
| **Reactome** | Another pathway database with more detailed biochemical pathway maps |
| **adjusted p-value** | FDR-corrected p-value — essential when testing hundreds of pathways simultaneously |
| **NES (Normalized Enrichment Score)** | GSEA score: positive = pathway genes are at the top of the ranked list (upregulated) |
| **leading edge** | The subset of pathway genes most responsible for the enrichment signal in GSEA |

## 🎯 Predict Before Running

1. You find 200 DE genes. 10 of them are in the "Cell Cycle" pathway, which has 50 genes total. The genome has ~20,000 genes. Is the Cell Cycle pathway enriched? Calculate the p-value conceptually.
2. What is the difference between Gene Ontology (GO) terms and KEGG pathways?
3. Why might a pathway enrichment analysis give you misleading results if your background gene set is wrong?


## 🟢 Complete Beginner's Guide

**Assumed background:** Zero knowledge of biological pathways or statistics required.

### What you need to know before starting

- **Biological pathway** — a series of molecular events in a cell (e.g. 'glycolysis' is the pathway that converts glucose to energy).
- **GO term (Gene Ontology term)** — a standardized label describing what a gene does. Example: GO:0006096 means 'glycolytic process'. Organized into three categories: Biological Process, Molecular Function, Cellular Component.
- **Gene set** — a curated list of genes known to participate in a specific process or pathway.
- **Enrichment analysis** — given a list of differentially expressed genes, test whether any pathway is over-represented compared to what you'd expect by chance. It answers: 'Are my changed genes clustered in one functional category?'
- **Fisher's exact test** — the statistical test most commonly used for enrichment. It tests whether the overlap between your gene list and a gene set is larger than random.

### Vocabulary you MUST know

| Term | One-line definition |
|------|--------------------|
| `background set` | The universe of all genes tested (not just significant ones) |
| `enrichment score` | How much more a pathway appears than expected |
| `FDR / Benjamini-Hochberg` | Multiple testing correction (you test thousands of pathways) |
| `KEGG` | Database of manually curated metabolic and signaling pathways |
| `MSigDB` | Molecular Signatures Database — comprehensive gene set collection |
| `ORA vs GSEA` | Over-Representation Analysis (threshold-based) vs Gene Set Enrichment Analysis (ranked list, no threshold) |

### 3-Step Reading Strategy for Beginners

1. **Read all markdown cells first** — grasp the biological motivation before the statistics.
2. **Run code cells one at a time** — examine the gene lists and pathway tables at each step.
3. **Modify one number and re-run** — change the p-value cutoff for enrichment and see which pathways appear or disappear.

### If you're stuck

- **YouTube:** 'Gene Set Enrichment Analysis (GSEA) explained' by StatQuest.
- **YouTube:** 'GO annotation and pathway analysis' by Bioinformatics Coach.
- **Book:** *Bioinformatics Data Skills*, Chapter 9.
- **Web:** https://www.geneontology.org/docs/go-enrichment-analysis/ — official GO enrichment guide.


## 📖 Prerequisites & Cross-References

**Before this notebook, you should be comfortable with:**
- **Differential expression** — producing gene lists ranked by significance (source of input for enrichment). *Review: `02_genomics_gene_analysis/02_rnaseq_analysis.ipynb`*
- **Statistics basics** — p-values, multiple testing correction, hypergeometric distribution. *Review: `00_python_ml_basics/05_hackerrank_statistics_ml_tracks.ipynb`*

**Quick recap of terms used in this notebook:**
- **pathway** — a set of genes working together in a biological process (e.g. "cell cycle", "apoptosis")
- **gene ontology (GO)** — standardised vocabulary categorising gene functions into molecular function, biological process, and cellular component
- **hypergeometric test** — probability of drawing k or more successes in n draws without replacement (like Fisher's exact test for enrichment)
- **fold enrichment** — (observed / expected) ratio; fold > 1 means the pathway is over-represented in your gene list


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.stats import hypergeom, fisher_exact
from collections import defaultdict

np.random.seed(42)

# ── SIMULATE GENE UNIVERSE WITH PATHWAY ANNOTATIONS ──
n_genes_total = 15000  # expressed genes in our experiment

# Define some "pathways" (gene sets)
pathways = {
    'Cell Cycle': list(range(0, 120)),
    'DNA Repair': list(range(50, 150)),         # overlaps Cell Cycle
    'Apoptosis': list(range(200, 300)),
    'PI3K-Akt Signaling': list(range(300, 480)),
    'MAPK Signaling': list(range(400, 550)),    # overlaps PI3K-Akt
    'Immune Response': list(range(600, 720)),
    'Metabolism': list(range(800, 1100)),
    'Translation': list(range(1200, 1340)),
    'Transcription': list(range(1400, 1600)),
    'Transport': list(range(2000, 2150)),
}

gene_names = [f'GENE_{i:05d}' for i in range(n_genes_total)]

# Simulate DE gene list: truly enriched in Cell Cycle, Apoptosis, and Immune Response
n_de = 300
de_genes = set()

# 30 from Cell Cycle (enriched)
de_genes.update(np.random.choice(pathways['Cell Cycle'], 30, replace=False))
# 25 from Apoptosis (enriched)
de_genes.update(np.random.choice(pathways['Apoptosis'], 25, replace=False))
# 20 from Immune Response (enriched)
de_genes.update(np.random.choice(pathways['Immune Response'], 20, replace=False))
# Rest: random background
remaining = list(set(range(n_genes_total)) - de_genes)
de_genes.update(np.random.choice(remaining, n_de - len(de_genes), replace=False))
de_genes = list(de_genes)[:n_de]

print(f"Gene universe: {n_genes_total} genes")
print(f"DE genes: {n_de}")
print(f"Pathways defined: {len(pathways)}")
for name, genes in pathways.items():
    overlap = len(set(genes) & set(de_genes))
    print(f"  {name:<30}: {len(genes):4d} genes, {overlap:3d} in DE list")

> **Expected output:** Gene universe size, DE gene count, and number of pathways defined  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

In [ ]:
# ── HYPERGEOMETRIC TEST FOR PATHWAY ENRICHMENT ──
# Model: drawing DE genes from the genome like colored balls from an urn
# H₀: DE genes are drawn uniformly at random (no pathway preference)
# Hypergeometric p-value = P(seeing >= k DE genes in a pathway of size M, by chance)

def hypergeometric_enrichment(pathway_genes, de_gene_set, background_size):
    """
    Parameters:
        pathway_genes: list of gene indices in the pathway
        de_gene_set: set of DE gene indices
        background_size: total number of tested genes
    Returns:
        p-value (one-sided, upper tail)
    """
    N = background_size              # total genes
    K = len(pathway_genes)           # genes in pathway
    n = len(de_gene_set)             # DE genes
    k = len(set(pathway_genes) & de_gene_set)  # DE genes in pathway

    if k == 0:
        return 1.0

    # hypergeom.sf(k-1, N, K, n) = P(X >= k)
    pval = hypergeom.sf(k - 1, N, K, n)

    # Fold enrichment
    expected = n * K / N
    fold_enrich = k / expected if expected > 0 else 0

    return pval, k, K, fold_enrich

# Test all pathways
enrichment_results = []
de_set = set(de_genes)

for pathway_name, pathway_gene_list in pathways.items():
    pval, k, K, fe = hypergeometric_enrichment(pathway_gene_list, de_set, n_genes_total)
    enrichment_results.append({
        'Pathway': pathway_name,
        'n_pathway': K,
        'n_overlap': k,
        'fold_enrichment': fe,
        'pvalue': pval
    })

results = pd.DataFrame(enrichment_results).sort_values('pvalue')

# BH FDR correction
from scipy.stats import rankdata
def bh_correct(pvalues):
    n = len(pvalues)
    order = np.argsort(pvalues)
    ranks = rankdata(pvalues)
    adjusted = np.minimum(1, pvalues * n / ranks)
    # Enforce monotonicity
    min_adj = adjusted.copy()
    for i in range(n-2, -1, -1):
        min_adj[order[i]] = min(min_adj[order[i]], min_adj[order[i+1]])
    return min_adj

results['padj'] = bh_correct(results['pvalue'].values)
results['significant'] = results['padj'] < 0.05

print("PATHWAY ENRICHMENT RESULTS:")
print(results[['Pathway','n_pathway','n_overlap','fold_enrichment','pvalue','padj','significant']].to_string(index=False))

print()
print("ANSWER TO PREDICT Q1:")
print("  k=10 DE genes in Cell Cycle (K=50), n=200 DE total, N=20000 genome")
pval_ex = hypergeom.sf(9, 20000, 50, 200)
expected_ex = 200 * 50 / 20000
print(f"  Expected by chance: {expected_ex:.1f} genes")
print(f"  Observed: 10 genes (fold enrichment: {10/expected_ex:.1f}x)")
print(f"  Hypergeometric p-value: {pval_ex:.2e}")

print()
print("ANSWER TO PREDICT Q2:")
print("  GO terms: hierarchical DAG of molecular function, biological process, cellular component")
print("  KEGG pathways: manually curated biochemical pathway maps with enzyme reactions")
print("  GO: broad, function-focused; KEGG: specific, mechanistic pathway diagrams")
print()
print("ANSWER TO PREDICT Q3:")
print("  If you use ALL human genes as background but only tested 8,000 expressed genes,")
print("  'background_size' is inflated → enrichment scores are underestimated.")
print("  CORRECT background: only genes you actually tested (expressed in your experiment).")

In [ ]:
# ── BUBBLE PLOT OF PATHWAY ENRICHMENT ──
sig = results[results['pvalue'] < 0.2].copy()
sig = sig.sort_values('fold_enrichment', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = [cm.RdYlGn(1 - p/sig['pvalue'].max()) for p in sig['pvalue']]
scatter = ax.scatter(sig['fold_enrichment'],
                     range(len(sig)),
                     s=sig['n_overlap'] * 20,
                     c=-np.log10(sig['pvalue'] + 1e-10),
                     cmap='RdYlGn',
                     vmin=0, vmax=5,
                     zorder=5)
ax.set_yticks(range(len(sig)))
ax.set_yticklabels(sig['Pathway'])
ax.set_xlabel('Fold Enrichment')
ax.set_title('Pathway Enrichment Analysis
(dot size = number of DE genes in pathway)')
ax.axvline(1, color='gray', linestyle='--', alpha=0.5)
cbar = plt.colorbar(scatter, ax=ax, label='-log₁₀(p-value)')
plt.tight_layout()
plt.savefig('pathway_enrichment.png', dpi=100)
plt.show()

print("KEY INTERPRETATION RULES:")
print("  1. Fold enrichment > 1: pathway is over-represented in DE genes")
print("  2. Large bubble = more DE genes hit this pathway")
print("  3. Dark color = more statistically significant")
print("  4. Watch out for: small pathways with high fold enrichment (1/2 genes hit = 2× = trivial)")
print("  5. Related pathways often both hit: PI3K-Akt and MAPK are both 'growth signaling'")
print()
print("PRODUCTION TOOLS:")
print("  - gseapy: Python wrapper for GSEA + over-representation analysis")
print("       pip install gseapy")
print("       import gseapy as gp")
print("       gp.enrichr(gene_list=de_gene_names, gene_sets='KEGG_2021_Human', ...)")
print("  - g:Profiler API: web service, also has Python client")
print("  - clusterProfiler: best tool (R), accessible via rpy2 from Python")

---
## 📚 Resources — Pathway Analysis & Functional Genomics

### University Courses (All Free)
| Course | Key Content |
|--------|-------------|
| **[MIT 7.91J Computational Systems Biology](https://ocw.mit.edu/courses/7-91j-foundations-of-computational-and-systems-biology-spring-2014/)** | Lecture 11-12: network biology, pathway enrichment |
| **[Harvard STAT115](https://canvas.harvard.edu/courses/39391)** | Bioinformatics analysis; pathway analysis lectures |
| **[StatQuest Pathway Analysis Series](https://www.youtube.com/playlist?list=PLblh5JKOoLUJo2Q6xK4tZElbIvAACEykp)** | GSEA, ORA, pathway analysis explained visually |

### Essential Reading (Free)
- **[GSEA paper](https://www.pnas.org/doi/10.1073/pnas.0506580102)** (Subramanian et al. 2005) — the foundational paper; explains why rank-based tests outperform ORA
- **[Pathway analysis is hard](https://academic.oup.com/bib/article/18/5/712/2562765)** (Khatri et al. 2012) — honest review of what pathway analysis can and cannot tell you; read before reporting results

### The Three Approaches (Know the Differences)
| Method | Input | Test | Pros | Cons |
|--------|-------|------|------|------|
| **ORA** (hypergeometric) | Gene list (DE genes) | Fisher/hypergeometric | Simple, fast | Threshold-dependent; loses magnitude |
| **GSEA** | Ranked gene list | KS-test on ranks | Uses all genes; threshold-free | Computationally expensive |
| **SPIA** | DE genes + fold change | Topology-aware | Uses pathway structure | Needs pathway topology |

### Key Pitfalls (Taught at Harvard MCB)
1. **Wrong gene universe**: your universe must be "all expressed genes", not "all human genes"
2. **Redundant terms**: GO has many redundant terms; use REVIGO to collapse them
3. **Publication bias**: famous pathways (p53, MAPK) appear enriched partly because they're overstudied
4. **Circular reasoning**: don't use pathway enrichment to "validate" a result if the genes were chosen using the same pathway

### Free Databases for Real Analysis
- **[MSigDB](https://www.gsea-msigdb.org/gsea/msigdb)** — curated gene sets for GSEA; most comprehensive
- **[Reactome](https://reactome.org/)** — manually curated pathway database; better quality than KEGG
- **[STRING](https://string-db.org/)** — protein-protein interaction networks; enrichment with network context


## 🏁 Checkpoint -- Pause and Verify

Before continuing, make sure you can:
1. Perform a Fisher's exact test for pathway enrichment given a gene list and background set
2. Explain the difference between ORA (threshold-based) and GSEA (ranked, no threshold)
3. Apply multiple testing correction (Benjamini-Hochberg FDR) and explain why it is necessary

**If any of these feel shaky**, re-read the section above or review the prerequisite notebook listed in the Cross-References section.

---
## 🎯 Interview Questions — Pathway Analysis

**Q1:** What is the difference between ORA (over-representation analysis) and GSEA?
> **A:** ORA takes a list of "significant" genes (requires a threshold) and tests if a pathway is overrepresented using a hypergeometric test. GSEA takes ALL genes ranked by a metric (e.g., fold change, -log p-value) and tests if a gene set members cluster at the top or bottom of the ranked list. GSEA is preferred because it uses the full signal without an arbitrary cutoff.

**Q2:** Your pathway analysis shows "ribosome biogenesis" as top hit. Is this biologically meaningful?
> **A:** Usually not. Ribosome biogenesis appears as a false positive whenever cells are proliferating, stressed, or if there's residual rRNA contamination. It's a red flag for data quality issues. Always check cell-cycle genes as a confound.

**Q3:** How would you connect pathway enrichment results to drug target identification?
> **A:** Find pathways differentially active in disease vs healthy. Within those pathways, identify druggable proteins (enzymes, receptors, kinases). Cross-reference with human genetic evidence (variants in pathway genes that modify disease risk). Then use structural biology (AlphaFold) to assess target tractability. This is a key workflow at drug discovery companies.


---
## 🔗 Cross-Module Connections

This notebook connects to:
- **Module 02/02 rnaseq_analysis (the gene list to enrich)**
- **Module 02/03 variant_analysis (gene sets from GWAS hits)**
- **Module 04/01 ml_for_omics (pathway features for ML models)**


**Progression:** After mastering this notebook, proceed to `04/01_ml_for_omics.ipynb`
to see how these features feed into ML models.


---
## ✅ Mastery Check — Pathway Enrichment

Before moving on, you should be able to answer these without looking:

1. Explain the difference between ORA and GSEA
2. State the hypergeometric test null hypothesis for ORA
3. Name 3 common false-positive pathways and why they appear
4. Explain why gene universe choice matters
5. Describe how pathway enrichment connects to drug target identification


**Score yourself:**
- 5/5: You're ready to move on
- 3-4/5: Review the cells you couldn't answer, then re-attempt
- < 3/5: Re-read the MIT 7.91J lecture on this topic, then redo all cells

**Time estimate:** If these questions take > 10 minutes each, revisit the notebook.
At interview pace, you should answer in 2-3 minutes per question.


## 📦 Real Datasets for Pathway Enrichment Analysis

### Dataset 1: Gene Ontology Annotation File (GAF format)
- **URL:** http://current.geneontology.org/annotations/goa_human.gaf.gz
- **Format:** Tab-separated GAF 2.2 format (gzip compressed)
- **Size:** ~50 MB compressed, ~300 MB uncompressed
- **Why it matters:** This is the definitive human gene-GO term mapping, curated by the Gene Ontology Consortium. Required for any real ORA or GSEA analysis.

### Dataset 2: MSigDB Hallmark Gene Sets
- **URL:** https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/h.all.v2023.2.Hs.json
- **Format:** JSON (gene sets as lists of HGNC gene symbols)
- **Size:** ~1 MB
- **Why it matters:** MSigDB Hallmarks are the 50 most well-defined biological processes — gold standard for pathway enrichment benchmarking.

### Dataset 3: KEGG Pathway API
- **URL:** https://rest.kegg.jp/list/pathway/hsa (REST API, no auth required)
- **Format:** Tab-separated pathway ID and name
- **Why it matters:** KEGG provides manually curated metabolic and signaling pathways. The REST API is free and requires no registration.

_Note: If downloads fail, the simulated gene sets in this notebook still teach the same enrichment analysis concepts._


In [ ]:
import requests
import json
import gzip
import os
import pandas as pd

# ── Dataset 1: MSigDB Hallmark Gene Sets (small, fast download) ──────────
MSIGDB_URL = (
    'https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2023.2.Hs/'
    'h.all.v2023.2.Hs.json'
)
try:
    print('Downloading MSigDB Hallmark gene sets...')
    response = requests.get(MSIGDB_URL, timeout=30)
    response.raise_for_status()
    msigdb = response.json()
    # Convert to dict: {pathway_name: [gene1, gene2, ...]}
    hallmarks = {entry['systematicName']: entry['geneSymbols']
                 for entry in msigdb['MSIGDB_XML_RECORDS']}
    print(f'Loaded {len(hallmarks)} hallmark gene sets')
    print('Example:', list(hallmarks.keys())[:3])
except Exception as e:
    print(f'MSigDB download failed: {e}')
    print('Using simulated gene sets instead.')
    hallmarks = {
        'HALLMARK_GLYCOLYSIS': ['HK1','HK2','PFKL','ALDOA','GAPDH','PGK1'],
        'HALLMARK_APOPTOSIS': ['CASP3','CASP8','BAX','BCL2','TP53','PUMA'],
        'HALLMARK_MYC_TARGETS_V1': ['MYC','MAX','CDK4','CCND1','E2F1'],
    }

# ── Dataset 2: KEGG Pathway List (human) ────────────────────────────────
KEGG_URL = 'https://rest.kegg.jp/list/pathway/hsa'
try:
    print('\nFetching KEGG human pathway list...')
    r = requests.get(KEGG_URL, timeout=30)
    r.raise_for_status()
    kegg_lines = r.text.strip().split('\n')
    kegg_pathways = [line.split('\t') for line in kegg_lines]
    kegg_df = pd.DataFrame(kegg_pathways, columns=['pathway_id', 'pathway_name'])
    print(f'Loaded {len(kegg_df)} KEGG human pathways')
    print(kegg_df.head())
except Exception as e:
    print(f'KEGG API failed: {e}')
    print('KEGG REST API: https://rest.kegg.jp — free, no auth needed')

# ── Dataset 3: GO Annotation File (GAF) — large file, optional ──────────
GOA_URL = 'http://current.geneontology.org/annotations/goa_human.gaf.gz'
GOA_LOCAL = '/tmp/goa_human.gaf.gz'
print('\nGO Annotation File (human):')
print(f'  URL: {GOA_URL}')
print(f'  To download: wget -O {GOA_LOCAL} {GOA_URL}')
print('  Format: GAF 2.2 (tab-separated, lines starting with ! are comments)')
print('  Size: ~50 MB compressed')
print('  Load example (after download):')
print('    import pandas as pd, gzip')
print('    df = pd.read_csv(GOA_LOCAL, sep="\\t", comment="!",')
print('         header=None, compression="gzip")')
print('    # Column 2=gene symbol, Column 4=GO ID, Column 8=aspect (P/F/C)')


---
## 🛠️ Exercise — Implement GSEA Preranked

**Goal:** Implement a simplified GSEA using a pre-ranked gene list.

GSEA ranks all genes by fold change, then tests if pathway genes cluster at the top or bottom using a running-sum statistic (Kolmogorov-Smirnov-like).

**Complete the `gsea_preranked()` function below. Expected: ES > 0 (pathway is upregulated).**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

def gsea_preranked(ranked_foldchanges, set_positions, p=1.0):
    # ranked_foldchanges: array sorted descending (most upregulated first)
    # set_positions: positions of pathway genes in the ranked list
    n = len(ranked_foldchanges)
    in_set = np.zeros(n, dtype=bool)
    in_set[set_positions] = True

    # TODO 1: weights[i] = abs(ranked_foldchanges[i])^p  if in_set[i], else 0
    weights = np.zeros(n)
    # YOUR CODE HERE

    # TODO 2: N_R = sum of all weights (normalization constant)
    N_R = 1.0  # FIX THIS

    n_set = in_set.sum()
    p_miss_inc = 1.0 / (n - n_set)

    # TODO 3: compute running_sum array
    running_sum = np.zeros(n)
    rs = 0.0
    for i in range(n):
        if in_set[i]:
            rs += weights[i] / N_R
        else:
            rs -= p_miss_inc
        running_sum[i] = rs

    ES = running_sum[np.argmax(np.abs(running_sum))]
    peak_idx = np.argmax(np.abs(running_sum))
    return {"ES": ES, "running_sum": running_sum, "peak_idx": peak_idx}

# Simulate: 30 pathway genes are upregulated (near top of ranking)
n_genes = 500
fc = np.random.randn(n_genes)
fc[:30] += 2.0  # pathway genes upregulated
ranked_idx = np.argsort(fc)[::-1]  # indices sorted by fc descending
ranked_fc = fc[ranked_idx]
pathway_positions = np.where(np.isin(ranked_idx, np.arange(30)))[0]

result = gsea_preranked(ranked_fc, pathway_positions)
print(f"Enrichment Score: {result['ES']:.4f}")
print(f"Peak at rank:     {result['peak_idx']} / {n_genes}")
print(f"Test: ES > 0 (pathway upregulated) -> {result['ES'] > 0}")

plt.figure(figsize=(8,3))
plt.plot(result["running_sum"], "b-", lw=1.5)
plt.axhline(0, color="k", ls="--", alpha=0.5)
plt.axvline(result["peak_idx"], color="r", ls=":", label=f'Peak ES={result["ES"]:.3f}')
plt.xlabel("Gene rank (0=most upregulated)")
plt.ylabel("Running Enrichment Score")
plt.title("GSEA Running Sum")
plt.legend()
plt.tight_layout()
plt.show()

# SOLUTION
print()
print("SOLUTION:")
print("  weights[in_set] = np.abs(ranked_foldchanges[in_set]) ** p")
print("  N_R = weights.sum()")


---
## 🐛 Debug Exercise — Fix the Pathway Enrichment Bugs

The code below has **3 bugs**. Find and fix all three.

**Expected:** Cell Cycle top hit with padj < 0.01, fold enrichment ~10x.

<details>
<summary>Show bugs and solutions</summary>

**Bug 1 (line: hypergeom.sf):** `hypergeom.sf(k, ...)` gives P(X > k). We want P(X >= k), so use `hypergeom.sf(k-1, N, K, n)`.

**Bug 2 (fold enrichment):** `k / K` ignores the background rate. Correct: `(k/n) / (K/N)` = `(k * N) / (n * K)`.

**Bug 3 (BH correction):** The cummin loop goes forward (`range(1, n)`) but should go backward (`range(n-2, -1, -1)`) to enforce monotonicity on the sorted adjusted p-values.

</details>


## Common Errors & Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `ModuleNotFoundError: No module named 'X'` | Package not installed | Run `!pip install X` in a cell, then restart kernel |
| `CUDA out of memory` | GPU RAM exceeded | Reduce batch size, or add `torch.cuda.empty_cache()` |
| `RuntimeError: Expected all tensors on same device` | Mixed CPU/GPU tensors | Add `.to(device)` after creating each tensor |
| `ValueError: shapes not aligned` | Matrix dimension mismatch | Print `tensor.shape` before the operation to debug |
| `KeyError` in DataFrame | Column name wrong or missing | Print `df.columns` to see exact column names |
| `IndexError: index out of range` | Loop or slice exceeds sequence length | Print `len(sequence)` and check your index |
| Kernel dies silently | Memory overflow (RAM) | Restart kernel, reduce data size, use generators |
| `UserWarning: No GPU found` | CUDA not available | Add `device = 'cuda' if torch.cuda.is_available() else 'cpu'` |

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import hypergeom
np.random.seed(42)

N = 15000; n_de = 300
cell_cycle = list(range(0, 120))
de_genes = set(list(range(0, 35)))
de_genes.update(np.random.choice(range(120, N), n_de - 35, replace=False))

def buggy_enrichment(pathway_genes, de_set, bg):
    K = len(pathway_genes); n = len(de_set)
    k = len(set(pathway_genes) & de_set)
    if k == 0: return 1.0, k, K, 0.0
    pval = hypergeom.sf(k, bg, K, n)          # BUG 1: should be sf(k-1, ...)
    fold_enrich = k / K                        # BUG 2: should be (k*bg)/(n*K)
    return pval, k, K, fold_enrich

def buggy_bh(pvalues):
    n_p = len(pvalues); order = np.argsort(pvalues)
    ranks = np.arange(1, n_p+1)
    adj = pvalues[order] * n_p / ranks
    for i in range(1, n_p):                    # BUG 3: should be range(n_p-2, -1, -1)
        adj[i] = min(adj[i], adj[i-1])
    out = np.empty(n_p); out[order] = adj
    return np.minimum(out, 1.0)

pathways = {"Cell Cycle": cell_cycle,
            "Random A": list(range(500,600)),
            "Random B": list(range(700,780))}
rows = []
for name, genes in pathways.items():
    pval, k, K, fe = buggy_enrichment(genes, de_genes, N)
    rows.append({"Pathway":name,"k":k,"K":K,"pval":pval,"FE":round(fe,2)})
df = pd.DataFrame(rows).sort_values("pval")
df["padj"] = buggy_bh(df["pval"].values)
print("BUGGY OUTPUT:")
print(df.to_string(index=False))
cc = df[df["Pathway"]=="Cell Cycle"].iloc[0]
print(f"\nCell Cycle: pval={cc['pval']:.4f}, padj={cc['padj']:.4f}, FE={cc['FE']}")
print(f"Is top hit: {df.iloc[0]['Pathway'] == 'Cell Cycle'}")
print("\nFix all 3 bugs so padj < 0.01 and FE > 5")
